# Insurance AI Model Fine-Tuning

This notebook fine-tunes open-source LLMs for insurance domain tasks using **Unsloth** + **LoRA**.

## Models:
1. **InsuranceChat** - Q&A about insurance policies, claims, underwriting
2. **DocumentGen** - Generate insurance clauses and policy documents

## Requirements:
- Google Colab with GPU (T4 free tier works!)
- ~2-4 hours for full training

## Quick Start:
1. Enable GPU: Runtime > Change runtime type > T4 GPU
2. Run all cells
3. Download trained model

In [ ]:
#@title Install Dependencies (run first)
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

In [ ]:
#@title Configuration
import os

# Model selection (phi3-mini is fastest, mistral-7b is best quality)
BASE_MODEL = "unsloth/Phi-3-mini-4k-instruct-bnb-4bit"  #@param ["unsloth/Phi-3-mini-4k-instruct-bnb-4bit", "unsloth/Phi-3.5-mini-instruct-bnb-4bit", "unsloth/mistral-7b-instruct-v0.3-bnb-4bit", "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"]

# Training type
MODEL_TYPE = "chat"  #@param ["chat", "docgen"]

# Training parameters
MAX_SAMPLES = 10000  #@param {type:"integer"}
NUM_EPOCHS = 1  #@param {type:"integer"}
BATCH_SIZE = 2  #@param {type:"integer"}
LEARNING_RATE = 2e-4  #@param {type:"number"}

# LoRA config
LORA_R = 16  #@param {type:"integer"}
LORA_ALPHA = 16  #@param {type:"integer"}
MAX_SEQ_LENGTH = 2048  #@param {type:"integer"}

print(f"Config: {BASE_MODEL}, {MODEL_TYPE}, {MAX_SAMPLES} samples, {NUM_EPOCHS} epochs")

In [ ]:
#@title Upload Training Data
from google.colab import files
import json

print("Upload your training data (JSONL format):")
print("- For chat: insurance_qa_train.jsonl")
print("- For docgen: clause_generation_train.jsonl")
print("")

uploaded = files.upload()

# Get uploaded filename
if uploaded:
    DATA_FILE = list(uploaded.keys())[0]
    print(f"\nUploaded: {DATA_FILE}")
    
    # Count examples
    with open(DATA_FILE, 'r') as f:
        count = sum(1 for _ in f)
    print(f"Total examples: {count}")
else:
    print("No file uploaded. Using sample data for testing.")
    DATA_FILE = None

In [ ]:
#@title Load Model
from unsloth import FastLanguageModel
import torch

print(f"Loading model: {BASE_MODEL}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
)

# Apply LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print("Model loaded with LoRA!")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
#@title Prepare Dataset
from datasets import load_dataset, Dataset
from unsloth.chat_templates import get_chat_template

# Setup chat template
chat_template = "phi-3" if "phi" in BASE_MODEL.lower() else "llama-3.1"
tokenizer = get_chat_template(tokenizer, chat_template=chat_template)

# Load data
if DATA_FILE and os.path.exists(DATA_FILE):
    print(f"Loading data from {DATA_FILE}")
    dataset = load_dataset("json", data_files=DATA_FILE, split="train")
else:
    print("Creating sample dataset for testing...")
    sample_data = [
        {
            "messages": [
                {"role": "system", "content": "You are an expert insurance professional."},
                {"role": "user", "content": "What is a deductible?"},
                {"role": "assistant", "content": "A deductible is the amount you pay out of pocket before your insurance coverage kicks in. For example, if you have a $500 deductible and file a $2,000 claim, you pay $500 and your insurance pays $1,500."}
            ]
        },
        {
            "messages": [
                {"role": "system", "content": "You are an expert insurance professional."},
                {"role": "user", "content": "What is the difference between term and whole life insurance?"},
                {"role": "assistant", "content": "Term life insurance provides coverage for a specific period (e.g., 20 years) and pays a death benefit only if you die during that term. Whole life insurance provides permanent coverage for your entire life and includes a cash value component that grows over time."}
            ]
        }
    ] * 50
    dataset = Dataset.from_list(sample_data)

# Limit samples
if MAX_SAMPLES and len(dataset) > MAX_SAMPLES:
    dataset = dataset.select(range(MAX_SAMPLES))

print(f"Dataset size: {len(dataset)} examples")

# Format for training
def format_conversation(example):
    if MODEL_TYPE == "chat":
        messages = example.get("messages", [])
    else:
        # DocGen format
        system = "You are an expert insurance document generator."
        messages = [
            {"role": "system", "content": system},
            {"role": "user", "content": example.get("prompt", "")},
            {"role": "assistant", "content": example.get("completion", "")}
        ]
    
    formatted = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": formatted}

dataset = dataset.map(format_conversation, remove_columns=dataset.column_names)
print("Dataset formatted!")
print(f"\nSample:\n{dataset[0]['text'][:500]}...")

In [ ]:
#@title Train Model
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        output_dir="outputs",
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=4,
        warmup_ratio=0.03,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        save_strategy="epoch",
        optim="adamw_8bit",
        weight_decay=0.001,
        lr_scheduler_type="cosine",
        seed=42,
        report_to="none",
    ),
)

print("Starting training...")
print(f"Estimated time: {len(dataset) * NUM_EPOCHS / (BATCH_SIZE * 4) / 60:.0f} minutes")

trainer_stats = trainer.train()

print(f"\nTraining complete!")
print(f"Final loss: {trainer_stats.training_loss:.4f}")

In [ ]:
#@title Save Model
import json
from datetime import datetime

# Save LoRA adapter
print("Saving LoRA adapter...")
model.save_pretrained("insurance_model_lora")
tokenizer.save_pretrained("insurance_model_lora")

# Save merged model (optional, takes more space)
SAVE_MERGED = True  #@param {type:"boolean"}
if SAVE_MERGED:
    print("Saving merged model (this may take a while)...")
    model.save_pretrained_merged(
        "insurance_model_merged",
        tokenizer,
        save_method="merged_16bit"
    )

# Save training info
info = {
    "model_type": MODEL_TYPE,
    "base_model": BASE_MODEL,
    "training_samples": len(dataset),
    "num_epochs": NUM_EPOCHS,
    "final_loss": float(trainer_stats.training_loss),
    "timestamp": datetime.now().isoformat()
}

with open("training_info.json", "w") as f:
    json.dump(info, f, indent=2)

print("\nModel saved!")
print("- LoRA adapter: insurance_model_lora/")
if SAVE_MERGED:
    print("- Merged model: insurance_model_merged/")

In [ ]:
#@title Test Model
from unsloth import FastLanguageModel

# Enable inference mode
FastLanguageModel.for_inference(model)

# Test questions
test_questions = [
    "What is the difference between actual cash value and replacement cost?",
    "Explain what reinsurance is and why companies use it.",
    "What does a standard homeowners policy cover?"
]

system_prompt = "You are an expert insurance professional with deep knowledge of insurance concepts."

for question in test_questions:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question}
    ]
    
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")
    
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=256,
        temperature=0.7,
        do_sample=True,
    )
    
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    
    print(f"Q: {question}")
    print(f"A: {response}")
    print("-" * 50)

In [ ]:
#@title Download Model
from google.colab import files
import shutil

# Create zip files for download
print("Creating downloadable archives...")

# LoRA adapter (smaller, recommended)
shutil.make_archive("insurance_model_lora", "zip", "insurance_model_lora")
print("Created: insurance_model_lora.zip")

# Merged model (larger, easier to use)
if os.path.exists("insurance_model_merged"):
    shutil.make_archive("insurance_model_merged", "zip", "insurance_model_merged")
    print("Created: insurance_model_merged.zip")

print("\nDownload options:")
print("1. LoRA adapter (~50MB) - Smaller, requires base model")
print("2. Merged model (~7GB) - Larger, standalone")

# Download LoRA adapter
files.download("insurance_model_lora.zip")

In [ ]:
#@title (Optional) Upload to Hugging Face Hub
from huggingface_hub import login

HF_TOKEN = ""  #@param {type:"string"}
HF_REPO = "your-username/insurance-chat"  #@param {type:"string"}

if HF_TOKEN:
    login(token=HF_TOKEN)
    
    # Push to Hub
    model.push_to_hub(HF_REPO, tokenizer=tokenizer, save_method="merged_16bit")
    print(f"Model uploaded to: https://huggingface.co/{HF_REPO}")
else:
    print("Enter HF_TOKEN to upload to Hugging Face Hub")